<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session13_building_agents_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۳: ساخت ایجنت از صفر

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session13_building_agents.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai sentence-transformers hazm
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/store_tools.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json

import numpy as np
from hazm import Normalizer
from llm_utils import download_data
from sentence_transformers import SentenceTransformer
from store_tools import STORE_TOOLS, run_tool, tool_schema

chunks = [json.loads(line) for line in open(download_data("novinkala_chunks.jsonl"), encoding="utf-8")]
_normalizer = Normalizer(persian_numbers=False)
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
embedder = SentenceTransformer("intfloat/multilingual-e5-base")
chunk_vectors = embedder.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True)


def search_docs(query: str) -> list:
    """Search the Novin Kala policy documents (shipping, returns, warranty, payment, instalments,
    club, orders, account, installation, support, gift cards, corporate sales).

    Args:
        query: a short question or keywords in Persian, e.g. "شرایط خرید اقساطی".
    """
    q = embedder.encode([f"query: {_normalizer.normalize(query).translate(_DIGITS)}"], normalize_embeddings=True)[0]
    top = np.argsort(-(chunk_vectors @ q))[:3]
    return [{"source": chunks[i]["id"], "text": chunks[i]["text"]} for i in top]


TOOLS = [search_docs] + STORE_TOOLS
for fn in TOOLS:
    print(f"{fn.__name__:16} {tool_schema(fn)['function']['description'][:80]}")

In [ ]:
import time

AGENT_SYSTEM = """You are the shopping assistant of Novin Kala, an Iranian online store. Answer in Persian.
You can use tools. Work step by step:
- Store rules (shipping, returns, warranty, payment, instalments, ...) come ONLY from search_docs.
- Products, prices and stock come ONLY from search_products or get_product.
- Do every calculation with the calculator tool.
- Before each tool call, write one short sentence about what you are going to check.
- When you have everything you need, give a short, clear final answer with the numbers you used.
- If the information is not available, say so; never invent prices, rules or numbers."""


class Agent:
    def __init__(self, tools, system=AGENT_SYSTEM, max_steps=8):
        self.tools, self.system, self.max_steps = tools, system, max_steps
        self.schemas = [tool_schema(t) for t in tools]

    def run(self, task, verbose=True):
        messages = [{"role": "system", "content": self.system}, {"role": "user", "content": task}]
        trace, seen, start = [], set(), time.time()
        for step in range(1, self.max_steps + 1):
            message = client.chat.completions.create(model=MODEL, messages=messages, tools=self.schemas,
                                                     temperature=0).choices[0].message
            if message.content and verbose:
                print(f"[{step}] thought: {message.content.strip()[:200]}")
            if not message.tool_calls:
                trace.append({"step": step, "type": "answer"})
                return {"answer": message.content, "trace": trace, "steps": step,
                        "seconds": time.time() - start}
            messages.append(message.model_dump(exclude_none=True))
            for call in message.tool_calls:
                key = (call.function.name, call.function.arguments)
                if key in seen:                       # the same call again: the agent is going in circles
                    result = json.dumps({"error": "you already made exactly this call; use its result "
                                                  "or try something different"})
                else:
                    result = run_tool(call.function.name, call.function.arguments, self.tools)
                seen.add(key)
                trace.append({"step": step, "type": "tool", "name": call.function.name,
                              "arguments": call.function.arguments, "result": result[:300]})
                if verbose:
                    print(f"[{step}] action: {call.function.name}({call.function.arguments})")
                    print(f"     observation: {result[:160]}")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
        return {"answer": "متأسفانه نتوانستم در تعداد گام مجاز به پاسخ برسم.", "trace": trace,
                "steps": self.max_steps, "seconds": time.time() - start}


agent = Agent(TOOLS)

In [ ]:
import re

import pandas as pd

TASKS = [
    {"task": "پیش‌پرداخت خرید اقساطی یک گلکسی A55 چقدر است؟", "expect_number": 7_225_000,
     "expect_tools": {"search_docs", "calculator"}},
    {"task": "هزینه ارسال عادی برای سفارش ۸۰۰ هزار تومانی به آمل چقدر است؟", "expect_number": 45_000,
     "expect_tools": {"shipping_cost"}},
    {"task": "قیمت دو تا هدفون سونی XM5 با ارسال سریع به تهران جمعاً چقدر می‌شود؟", "expect_number": 43_885_000,
     "expect_tools": {"search_products", "shipping_cost"}},
    {"task": "گارانتی لپ‌تاپ IdeaPad چند ماه است؟", "expect_number": 18, "expect_tools": {"search_docs"}},
    {"task": "وضعیت سفارش NK482913 چیست؟", "expect_number": None, "expect_tools": {"track_order"}},
    {"task": "اگر ۳۰۰ امتیاز باشگاه داشته باشم، به چقدر اعتبار کیف پول تبدیل می‌شود؟", "expect_number": 1_500_000,
     "expect_tools": {"search_docs"}},
]


def numbers_in(text):
    text = (text or "").translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
    text = re.sub(r"(?<=\d)[,٬](?=\d{3})", "", text)
    return {int(float(n)) for n in re.findall(r"\d+(?:\.\d+)?", text)}


REFUSAL_WORDS = ["پیدا نکردم", "اطلاعاتی", "موجود نیست", "ندارد", "نمی‌دانم", "در دسترس نیست", "ذکر نشده"]


def evaluate_agent(agent, tasks=TASKS):
    rows = []
    for t in tasks:
        r = agent.run(t["task"], verbose=False)
        used = {s["name"] for s in r["trace"] if s["type"] == "tool"}
        if t.get("expect_refusal"):
            success = any(w in (r["answer"] or "") for w in REFUSAL_WORDS)
        else:
            success = t["expect_number"] is None or t["expect_number"] in numbers_in(r["answer"])
        rows.append({"task": t["task"][:45], "success": success, "tools_ok": t.get("expect_tools", set()) <= used,
                     "steps": r["steps"], "seconds": round(r["seconds"], 1)})
    return pd.DataFrame(rows)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

به ایجنت یک گام **برنامه‌ریزی** اضافه کنید: پیش از شروع حلقه، از مدل بخواهید فهرست گام‌ها را بنویسد و این برنامه را به پیام کاربر اضافه کنید. نرخ موفقیت و تعداد گام‌ها را با نسخهٔ بدون برنامه مقایسه کنید.

</div>

In [ ]:
class PlanningAgent(Agent):
    def run(self, task, verbose=True):
        plan = chat(client, MODEL,
                    f"Task: {task}\n\nAvailable tools: {', '.join(t.__name__ for t in self.tools)}.\n"
                    "Write a numbered plan of at most 4 short steps (which tool, for what). No answer yet.",
                    temperature=0)
        if verbose:
            print("PLAN:\n" + plan)
        return super().run(f"{task}\n\nPlan:\n{plan}", verbose=verbose)


summary = pd.concat({"no plan": evaluate_agent(Agent(TOOLS)), "with plan": evaluate_agent(PlanningAgent(TOOLS))})
summary.groupby(level=0)[["success", "tools_ok", "steps", "seconds"]].mean().round(2)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برنامه‌ریزی معمولاً دو اثر دارد: مسیر را منظم‌تر می‌کند، مثلاً ایجنت یادش نمی‌رود قاعده را از اسناد بگیرد، ولی یک فراخوانی اضافهٔ مدل هزینه دارد و اگر برنامه اشتباه باشد، ایجنت همان اشتباه را با اطمینان دنبال می‌کند. برای کارهای کوتاه مثل این مجموعه، اثر برنامه‌ریزی ممکن است کم یا حتی منفی باشد. برای کارهای طولانی‌تر با چند زیرهدف، فایده‌اش بیشتر است. با ۶ کار، نتیجه را با احتیاط تفسیر کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

تشخیص تکرار فعلی فقط فراخوانی **کاملاً یکسان** را می‌گیرد. آن را طوری گسترش دهید که اگر ایجنت سه بار پشت سر هم یک ابزار را صدا زد، حتی با آرگومان‌های متفاوت، هشدار بگیرد. کاری طراحی کنید که این حالت را نشان دهد.

</div>

In [ ]:
class LoopGuardAgent(Agent):
    """Warn the model when it calls the same tool three times in a row, even with different arguments."""

    def run(self, task, verbose=True):
        original = run_tool
        recent = []

        def guarded(name, arguments, tools):
            recent.append(name)
            result = original(name, arguments, tools)
            if len(recent) >= 3 and len(set(recent[-3:])) == 1:
                result = json.dumps({"warning": f"you called {name} three times in a row. Stop searching and "
                                                "answer with what you have, or explain what is missing.",
                                     "last_result": json.loads(result)}, ensure_ascii=False)
            return result

        globals()["run_tool"] = guarded
        try:
            return super().run(task, verbose)
        finally:
            globals()["run_tool"] = original


# A task that tempts the agent to keep searching: the product does not exist.
result = LoopGuardAgent(TOOLS).run("گوشی نوکیا 3310 قرمز دارید؟ اگه نه، نوکیا مدل دیگه چی، اونم نه؟ پس موتورولا؟")
print("\nFINAL:", result["answer"])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

جستجوی پشت‌سرهم با عبارت‌های کمی متفاوت («نوکیا 3310»، «نوکیا قرمز»، «نوکیا») الگوی رایج ایجنتی است که چیزی پیدا نمی‌کند و نمی‌خواهد تسلیم شود. تشخیص فراخوانی **یکسان** جلسه این را نمی‌گیرد. هشدار سه‌تایی به مدل می‌گوید توقف کند و با همان اطلاعات جواب دهد.

پیاده‌سازی ما `run_tool` را موقتاً عوض می‌کند تا کلاس `Agent` تغییر نکند. در کد واقعی، بهتر است کلاس از ابتدا یک «اجراکنندهٔ ابزار» قابل تعویض داشته باشد. این نمونه‌ای از اهمیت طراحی قابل توسعه است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

چهار کار جدید به `TASKS` اضافه کنید که دست‌کم دو تای آن‌ها پاسخی در اسناد ندارند و ایجنت باید بگوید اطلاعات موجود نیست. بررسی‌کنندهٔ مناسب برای این کارها بنویسید.

</div>

In [ ]:
NEW_TASKS = [
    {"task": "تخفیف خرید سازمانی ۳۰ میلیون تومانی چند درصد است؟", "expect_number": 3,
     "expect_tools": {"search_docs"}},
    {"task": "اگه سه تا شارژر انکر بخرم، با ارسال عادی به اصفهان جمعاً چقدر میشه؟", "expect_number": 5_700_000,
     "expect_tools": {"search_products"}},
    {"task": "نوین‌کالا در شیراز فروشگاه حضوری دارد؟ آدرسش کجاست؟", "expect_refusal": True},
    {"task": "قیمت لپ‌تاپ دست دوم ایسوس در نوین‌کالا چقدر است؟", "expect_refusal": True},
]
evaluate_agent(Agent(TOOLS), NEW_TASKS)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برای کار دوم: سه شارژر ۱٬۹۰۰٬۰۰۰ تومانی می‌شود ۵٬۷۰۰٬۰۰۰ تومان که بیش از یک میلیون است، پس ارسال عادی رایگان است و جمع کل همان ۵٬۷۰۰٬۰۰۰ تومان است. ایجنت باید این قاعده را بداند، چه از ابزار `shipping_cost` چه از اسناد.

برای کارهای بی‌پاسخ، بررسی‌کننده دنبال **عبارت‌های امتناع** می‌گردد. این بررسی ساده و شکننده است: ایجنت ممکن است بگوید «فقط فروش اینترنتی داریم» که درست است ولی هیچ‌کدام از کلمه‌های فهرست را ندارد. بررسی دقیق‌تر با LLM-as-a-Judge انجام می‌شود (جلسهٔ ۵): «آیا پاسخ بدون ساختن اطلاعات، می‌گوید این اطلاعات در دسترس نیست؟»

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

گردش کار ثابتی برای پرسش «هزینهٔ کل خرید n عدد از کالای X با روش ارسال Y به شهر Z» بنویسید. روی ۱۰ نمونه، دقت و زمان آن را با ایجنت مقایسه کنید.

</div>

In [ ]:
import random
import time

from store_tools import search_products, shipping_cost


def total_cost(product_query, n, method, city):
    """A fixed workflow: product price x n + shipping according to the policy."""
    product = search_products(product_query)[0]
    items = product["price_toman"] * n
    shipping = shipping_cost(items, method, city)
    if shipping.get("available") is False:
        return None
    return items + shipping["cost"]


random.seed(0)
cases = [(q, random.randint(1, 3), random.choice(["standard", "express"]), random.choice(["تهران", "آمل", "شیراز"]))
         for q in ["گلکسی A55", "هدفون سونی", "شارژر انکر", "پاوربانک شیائومی", "تلویزیون شیائومی 43"] * 2]

rows = []
for q, n, method, city in cases:
    start = time.time()
    expected = total_cost(q, n, method, city)
    workflow_seconds = time.time() - start
    method_fa = "ارسال عادی" if method == "standard" else "ارسال سریع"
    r = Agent(TOOLS).run(f"هزینه کل خرید {n} عدد {q} با {method_fa} به {city} چقدر است؟", verbose=False)
    ok = expected is None or expected in numbers_in(r["answer"])
    rows.append({"case": f"{n} x {q}, {method}, {city}", "expected": expected, "agent_ok": ok,
                 "agent_s": round(r["seconds"], 1), "workflow_s": round(workflow_seconds, 3)})
comparison = pd.DataFrame(rows)
print(f"agent accuracy: {comparison.agent_ok.mean():.0%}   "
      f"mean time: agent {comparison.agent_s.mean():.1f}s, workflow {comparison.workflow_s.mean():.3f}s")
comparison

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

گردش کار برای این نوع پرسش همیشه درست و تقریباً بی‌هزینه است، ولی فقط وقتی پرسش دقیقاً در این قالب باشد. نکتهٔ ظریف: ما پاسخ «درست» را با همان گردش کار حساب کردیم. این فقط وقتی معتبر است که گردش کار خودش تست شده باشد. در غیر این صورت، اشتباه گردش کار به شکل «اشتباه ایجنت» دیده می‌شود.

در محصول واقعی، این دو کنار هم کار می‌کنند. یک مسیریاب (جلسهٔ ۱۶) پرسش‌هایی را که با قالب گردش کار جور است، با پارامترهای استخراج‌شده به آن می‌فرستد و بقیه را به ایجنت.

</div>